# Démo : Reformulation des questions

**Contexte :** assistant du service client de Lagune Banque.

Les questions des clients ne reprennent pas toujours les termes des documents. Cette démo compare trois méthodes de reformulation avant la recherche (réécriture, multi-requête avec fusion RRF et HyDE) et le reranking après la recherche. Les résultats sont évalués sur des questions courantes et difficiles.

**Sommaire**
- [Configuration](#configuration)
- [Index et questions](#index-questions)
- [Recherche initiale](#recherche-initiale)
- [Réécriture](#reecriture)
- [Multi-requête](#multi-requete)
- [HyDE](#hyde)
- [Reranking](#reranking)
- [Évaluation](#evaluation)
- [Synthèse](#synthese)

```mermaid
flowchart LR
    Q["Question du client"] --> R["Réécriture : ra.rewrite"]
    Q --> M["Plusieurs requêtes : ra.variants"]
    M --> F["Fusion RRF : rrf"]
    Q --> H["HyDE : ra.hypothesis"]
    R --> S["Recherche"]
    F --> S
    H --> S
    S --> K["Reranking : rag.rerank"]
```

Ce notebook ajoute la deuxième capacité de l'assistant : la reformulation.

## <a id="configuration"></a>Configuration

In [36]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
import rag_advanced as ra
from config import prepare
from weaviate_store import connect

In [37]:
prepare(embedding=True, rerank=True)

Connexion OK : gemini/gemini-3.5-flash-lite gemini/gemini-embedding-2 azure_ai/Cohere-rerank-v4.0-fast


## <a id="index-questions"></a>Index et questions

L'index est celui retenu au TP1 : splitter récursif, 500 caractères, overlap 75. Les questions viennent de `data/evaluation/questions_rag.json` :

- 12 **cas courants** : des questions bien formulées ;
- 35 **cas difficiles** : fautes, anglais, nombres exacts, mots seuls, suites de conversation, et 15 questions générées par un modèle avec des mots éloignés de ceux des documents, que la recherche de base rate.

In [38]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
index = rag.index_chunks(connect(), "index_reformuler", rag.split_documents(documents, 500, 75, "reformuler"))

courants = ra.load_questions("representatif")
difficiles = ra.load_questions("difficile")
pd.Series([x["famille"] for x in difficiles]).value_counts().to_frame("cas difficiles")

Weaviate injoignable (WeaviateConnectionError) : repli sur la base en mémoire. Pour Weaviate : `docker compose up -d` à la racine du dépôt.


,cas difficiles
générée (échec du départ),15
bruitée,4
anglais,4
nombre exact,4
courte,4
suite de conversation,4


## <a id="recherche-initiale"></a>Recherche initiale

La fonction `display_table` affiche les trois premiers chunks pour une requête. Une question de suite, sans son contexte, ne veut rien dire pour la recherche.

In [39]:
def retrieve(requete, n=10):
    return rag.search(index, requete, n, rag.AUJOURDHUI)


def display_table(requete, n=3):
    for rang, p in enumerate(retrieve(requete, n), 1):
        print(f"[{rang}] {p['meta']['source']} | {p['texte'][:100]!r}")


ECHANGE = "Client : Quels sont les frais de retard du Prêt Express ?\nAssistant : 2 000 FCFA, en forfait."
display_table("Et pour l'Express+ ?")

[1] DOC-03 | 'Fiche produit – Prêt Express+\n\nIdentifiant : DOC-03 · Version 1.0 · En vigueur depuis le 01/01/2025 '
[2] DOC-01 | '4.2 Le plafond initial du Prêt Express est de 15 000 FCFA. Il augmente de 5 000\nFCFA après chaque sé'
[3] DOC-06 | '2 000 FCFA\nForfait\nPrêt Express+\n5 000 FCFA\nPar échéance impayée\nNote 3 – Les frais de retard sont a'


La recherche ne voit pas l'échange précédent. Le chunk qui répond (les frais de retard de l'Express+, dans la grille tarifaire) n'arrive qu'en troisième position, derrière la fiche produit et une règle de plafond.

## <a id="reecriture"></a>Réécriture

Un modèle de langage réécrit la question pour qu'elle se comprenne seule. `ra.rewrite` reçoit la question et l'échange précédent. Par défaut elle reste fidèle : elle corrige les fautes et résout les renvois, sans rien ajouter.

In [40]:
requete = ra.rewrite("Et pour l'Express+ ?", ECHANGE)
print(requete)
display_table(requete)

Quels sont les frais de retard du Prêt Express+ ?
[1] DOC-03 | "Retard\n\nEn cas d'échéance impayée, des frais de retard forfaitaires de 5 000 FCFA par échéance s'app"
[2] DOC-03 | '30 jours\n3 % du montant\n4 500 FCFA\n\n60 jours\n6 % du montant\n9 000 FCFA\n\n90 jours\n9 % du montant\n13 5'
[3] DOC-06 | '2 000 FCFA\nForfait\nPrêt Express+\n5 000 FCFA\nPar échéance impayée\nNote 3 – Les frais de retard sont a'


In [41]:
for question in ["fré de retar prèt expres ??", "c koi le plafon du pret express stp", "Quel est le montant minimum d'un Prêt Express ?"]:
    print(f"{question!r}  ->  {ra.rewrite(question)!r}")

'fré de retar prèt expres ??'  ->  'Frais de retard prêt express ?'
'c koi le plafon du pret express stp'  ->  "Quel est le plafond du prêt exprès s'il vous plaît ?"
"Quel est le montant minimum d'un Prêt Express ?"  ->  "Quel est le montant minimum d'un Prêt Express ?"


Avec la requête réécrite, le chunk sur les frais de retard passe en première position. Les fautes sont corrigées, et la question déjà claire n'est pas modifiée. Avec `mots_cles=True`, la réécriture enrichit au contraire la requête de synonymes et de termes de métier :

In [42]:
for question in ["Quels sont les frais de retard du Prêt Express ?", "Combien de temps avant de pouvoir réemprunter ?"]:
    print(f"{question!r}  ->  {ra.rewrite(question, mots_cles=True)!r}")

'Quels sont les frais de retard du Prêt Express ?'  ->  'frais retard Prêt Express pénalités impayé bancaire'
'Combien de temps avant de pouvoir réemprunter ?'  ->  "délai nouveau crédit prêt immobilier consommation réemprunter capacité d'emprunt"


Ces mots peuvent combler l'écart de vocabulaire avec la documentation, mais ils dégradent une question qui était bonne. La mesure plus bas départage les deux versions.

## <a id="multi-requete"></a>Multi-requête et fusion

La multi-requête demande au modèle trois reformulations, cherche avec chacune, puis fusionne les listes. La fusion **RRF** (Reciprocal Rank Fusion) donne à chaque chunk `1 / (60 + rang)` dans chaque liste et additionne : un chunk bien classé dans plusieurs listes remonte, sans qu'il faille comparer des distances de natures différentes.

`rrf` additionne ces scores par chunk (un chunk est identifié par son texte) et garde les `n` meilleurs.

In [43]:
def rrf(listes, k=60, n=10):
    scores, vus = {}, {}
    for liste in listes:
        for rang, p in enumerate(liste, 1):
            scores[p["texte"]] = scores.get(p["texte"], 0) + 1 / (k + rang)
            vus[p["texte"]] = p
    return [vus[t] for t, _ in sorted(scores.items(), key=lambda x: -x[1])[:n]]

In [44]:
question = "Quelle somme maximale puis-je obtenir au total ?"
listes = {"question d'origine": retrieve(question)}
for i, variante in enumerate(ra.variants(question), 1):
    listes[f"variante {i} : {variante[:55]}"] = retrieve(variante)
listes["fusion RRF"] = rrf(list(listes.values()))

pd.DataFrame({nom: [f"{p['meta']['source']} : {p['texte'][:38]!r}" for p in chunks[:3]] for nom, chunks in listes.items()}, index=["1er", "2e", "3e"]).T

,1er,2e,3e
question d'origine,DOC-06 : '30 000 FCFA\n8 à 9 prêts remboursés\...,DOC-01 : '4.2 Le plafond initial du Prêt Express',DOC-06 : 'Élément\nValeur\nDépôt minimum\n1 00...
variante 1 : Quel est le montant maximal total que je peux emprunter,DOC-06 : '30 000 FCFA\n8 à 9 prêts remboursés\...,DOC-01 : '4.2 Le plafond initial du Prêt Express',"DOC-01 : ""Article 3 – Conditions d'éligibilité..."
variante 2 : À combien s'élève le plafond global de crédit accordé ?,DOC-06 : '30 000 FCFA\n8 à 9 prêts remboursés\...,DOC-01 : '4.2 Le plafond initial du Prêt Express',"DOC-02 : ""Le montant d'un Prêt Express est compr"""
variante 3 : Quel est le crédit total maximum auquel j'ai droit ?,DOC-06 : '30 000 FCFA\n8 à 9 prêts remboursés\...,DOC-01 : '4.2 Le plafond initial du Prêt Express',"DOC-01 : ""Article 3 – Conditions d'éligibilité..."
fusion RRF,DOC-06 : '30 000 FCFA\n8 à 9 prêts remboursés\...,DOC-01 : '4.2 Le plafond initial du Prêt Express',"DOC-01 : ""Article 3 – Conditions d'éligibilité..."


## <a id="hyde"></a>HyDE

Une réponse ressemble plus au chunk cherché qu'une question. HyDE demande au modèle d'écrire une réponse plausible, même inventée, et cherche avec elle.

In [45]:
reponse = ra.hypothesis("Combien de temps avant de pouvoir réemprunter ?")
print(reponse)
display_table(reponse)

Après un remboursement anticipé ou total, vous pouvez soumettre une nouvelle demande de prêt dès l'enregistrement définitif de votre dossier, soit généralement sous 15 jours ouvrés. Ce délai peut toutefois varier en fonction de votre situation financière et nécessite un nouveau calcul de votre capacité d'emprunt, avec un minimum de 3 mois d'ancienneté pour les clients changeant d'établissement.
[1] DOC-01 | "Article 3 – Conditions d'éligibilité\n3.1 Pour demander un prêt, le client doit remplir les critères "
[2] DOC-01 | '4.2 Le plafond initial du Prêt Express est de 15 000 FCFA. Il augmente de 5 000\nFCFA après chaque sé'
[3] DOC-01 | "peut contacter le service client.\nArticle 10 – Suspension de l'accès à un nouveau prêt"


## <a id="reranking"></a>Reranking

Les méthodes précédentes changent la **requête**. Le reranking agit après la recherche : un modèle de reranking lit la question et chaque chunk ensemble, et note leur pertinence. Il est plus précis qu'un embedding, qui encode la question et le chunk séparément, mais trop lent pour parcourir toute la base. On procède en deux étapes : la recherche ramène 20 candidats, `rag.rerank` les reclasse et garde les 10 meilleurs.

In [46]:
def retrieve_rerank(requete, n=10):
    return rag.rerank(requete, retrieve(requete, 20), n)


for rang, p in enumerate(retrieve_rerank("Et pour l'Express+ ?", 3), 1):
    print(f"[{rang}] score {p['score']:.2f} | {p['meta']['source']} | {p['texte'][:100]!r}")

[1] score 0.78 | DOC-01 | '4.2 Le plafond initial du Prêt Express est de 15 000 FCFA. Il augmente de 5 000\nFCFA après chaque sé'
[2] score 0.77 | DOC-03 | 'Fiche produit – Prêt Express+\n\nIdentifiant : DOC-03 · Version 1.0 · En vigueur depuis le 01/01/2025 '
[3] score 0.73 | DOC-06 | ': frais de 9 000 FCFA\nPrêt Express+\n90 jours\n9 % du montant\nPrêt de 150 000 FCFA\n: frais de 13 500 F'


## <a id="evaluation"></a>Évaluation

Chaque méthode est appliquée aux 47 questions. Pour chacune, on cherche le rang du premier chunk qui contient la réponse complète. Le **MRR** est la moyenne de `1 / rang` : 1,00 si la bonne réponse est toujours en première position. Le reranking est mesuré seul, puis combiné à la réécriture fidèle.

La mesure dure quelques minutes : chaque technique appelle le modèle de langage, et le reranking appelle en plus le service de reranking pour chaque question.

In [47]:
METHODES = {
    "vectoriel seul": lambda q, h: retrieve(q),
    "réécriture fidèle": lambda q, h: retrieve(ra.rewrite(q, h)),
    "réécriture avec mots-clés": lambda q, h: retrieve(ra.rewrite(q, h, mots_cles=True)),
    "multi-requête (RRF)": lambda q, h: rrf([retrieve(v) for v in [q] + ra.variants(q)]),
    "HyDE": lambda q, h: retrieve(ra.hypothesis(q)),
    "reranking": lambda q, h: retrieve_rerank(q),
    "réécriture + reranking": lambda q, h: retrieve_rerank(ra.rewrite(q, h)),
}


def mrr(rangs):
    return round(sum(1 / r for r in rangs if r) / len(rangs), 2)


resultats = {}
for nom, methode in METHODES.items():
    resultats[nom] = {"courants": ra.measure(methode, courants), "difficiles": ra.measure(methode, difficiles)}
    print("fait :", nom)

fait : vectoriel seul
fait : réécriture fidèle
fait : réécriture avec mots-clés
fait : multi-requête (RRF)
fait : HyDE
fait : reranking
fait : réécriture + reranking


In [48]:
pd.DataFrame({
    nom: {
        "cas courants": mrr(r["courants"]["rangs"]),
        "cas difficiles": mrr(r["difficiles"]["rangs"]),
    } for nom, r in resultats.items()}).T

,cas courants,cas difficiles
vectoriel seul,0.80,0.62
réécriture fidèle,0.80,0.67
réécriture avec mots-clés,0.76,0.63
multi-requête (RRF),0.84,0.61
HyDE,0.86,0.64
reranking,0.92,0.74
réécriture + reranking,0.92,0.74


## <a id="synthese"></a>Synthèse

1 est le MRR maximal, et le vectoriel seul est la référence de chaque colonne. Les chiffres varient légèrement d'une exécution à l'autre ; l'ordre de grandeur reste le même.

- **La réécriture fidèle règle les suites de conversation** (de 0,7 à 1,0 environ) sans toucher aux cas courants. C'est la plus sûre.
- **La réécriture avec mots-clés aide les questions dont le vocabulaire diffère** des documents (les échecs du départ, de 0,30 à 0,4 environ), mais **dégrade les cas courants** (de 0,80 à 0,7 environ) : les mots ajoutés diluent une question qui était bonne.
- **La multi-requête et HyDE gagnent peu** sur les échecs du départ (jusqu'à 0,37 et 0,38) pour deux à quatre fois plus de temps. HyDE ne voit pas l'échange précédent : il est moins bon que le vectoriel seul sur les suites de conversation.
- **Aucune technique ne règle les échecs du départ** : ils restent sous 0,45. La reformulation ne suffit pas, il faut agir sur le classement des chunks.

| Technique | Aide quand | Nuit quand | Coût |
|---|---|---|---|
| Réécriture fidèle | la question dépend de l'échange précédent, ou contient des fautes | presque jamais | un appel au modèle |
| Réécriture avec mots-clés | le client n'emploie pas les mots de la documentation | la question était déjà claire | un appel au modèle |
| Multi-requête (RRF) | la formulation s'écarte de la documentation | le gain est faible face au temps ajouté | un appel au modèle et plusieurs recherches : environ 4 fois plus long |
| HyDE | le vocabulaire s'écarte de la documentation | la question dépend de l'échange précédent | un appel au modèle |